# ==============================================================================
# 🏡 SPATIAL GEOSPATIAL VALUATION: CALIFORNIA HOUSING VIA VOTING REGRESSOR
# ==============================================================================
### Core Theoretical Principles:
Combines three high-capacity non-linear ensembles:
1. `RandomForestRegressor`
2. `GradientBoostingRegressor`
3. `HistGradientBoostingRegressor`


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import VotingRegressor, RandomForestRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: California Spatial Housing Environment ready.")


✅ STEP 1: California Spatial Housing Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle California Housing dataset (20,640 census blocks).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/housing/housing.csv'
df = pd.read_csv(data_path)

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} census blocks, {df.shape[1]} attributes")
print(f"Median House Value Summary:\n{df['median_house_value'].describe().round(2)}")
df.head(3)


📊 Dataset Shape: 20,640 census blocks, 10 attributes
Median House Value Summary:
count     20640.00
mean     206855.82
std      115395.62
min       14999.00
25%      119600.00
50%      179700.00
75%      264725.00
max      500001.00
Name: median_house_value, dtype: float64


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `median_house_value`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['median_house_value'])
y = df['median_house_value']

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")


Missing attributes:
total_bedrooms    207
dtype: int64


In [4]:
# STEP 6: TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Blocks: {X_train.shape[0]:,}")
print(f"Test Blocks : {X_test.shape[0]:,}")


Train Blocks: 16,512
Test Blocks : 4,128


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Median imputation for `total_bedrooms` and One-Hot encoding for `ocean_proximity`.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['ocean_proximity']
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', SimpleImputer(strategy='median'), num_cols)
])
print("✅ STEP 7: Spatial Preprocessor configured.")


✅ STEP 7: Spatial Preprocessor configured.


## ⚠️ STEP 8: ENTERPRISE TRI-MODEL ENSEMBLE FIT
Fitting RF + GBM + HistGBM.


In [6]:
# STEP 8: ENSEMBLE FIT
vote_reg = VotingRegressor(
    estimators=[
        ('rf', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)),
        ('gbm', GradientBoostingRegressor(n_estimators=100, learning_rate=0.08, max_depth=4, subsample=0.8, random_state=42)),
        ('hgbm', HistGradientBoostingRegressor(max_iter=100, learning_rate=0.08, random_state=42))
    ],
    n_jobs=-1
)

pipe_voting = Pipeline([('prep', preprocessor), ('vote', vote_reg)])
pipe_voting.fit(X_train, y_train)

r2_vote = r2_score(y_test, pipe_voting.predict(X_test))
print(f"Enterprise Tri-Ensemble R²: {r2_vote * 100:.2f}%")


Enterprise Tri-Ensemble R²: 78.11%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing weights among the three high-performance tree ensembles.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'vote__weights': [
        [1, 1, 1],
        [1, 2, 1],
        [1, 2, 2]
    ]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_voting, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Property Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV R²: {grid.best_score_*100:.2f}%")


🏆 Best Property Hyperparameters: {'vote__weights': [1, 2, 2]}
🎯 Best 3-Fold CV R²: 79.78%


## 📊 STEP 10 & 11: VALUATION REPORT & RESIDUALS
Evaluating automated property valuation residuals.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("📋 CALIFORNIA PROPERTY VALUATION METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2 * 100:.2f}%")


📋 CALIFORNIA PROPERTY VALUATION METRICS:
   Mean Absolute Error (MAE) : $35,243.91
   Root Mean Squared Error   : $51,976.93
   R² Score                  : 79.38%


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & PROPERTY APPRAISAL SMOKE TEST
Deploying pipeline and validating automated valuation model (AVM) latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/california_housing_voting_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_property = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_val = loaded_pipe.predict(sample_property)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏡 LIVE INSTANT AUTOMATED VALUATION (AVM):")
print(f"   Estimated Median Value: 🏷️ ${pred_val:,.2f}")
print(f"   Actual Listed Value   : ${y_test.iloc[0]:,.2f}")
print(f"   Appraisal Latency     : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/california_housing_voting_pipeline.joblib (581,889 bytes)



🏡 LIVE INSTANT AUTOMATED VALUATION (AVM):
   Estimated Median Value: 🏷️ $58,332.87
   Actual Listed Value   : $47,700.00
   Appraisal Latency     : 9.211 ms (SLA < 2.0ms: CHECK)
